---
## Part 1: Combine State Shapefiles

The 2020-2021 validation data arrives as separate shapefiles per state (some states have multiple files). These are combined into a single GeoDataFrame before any processing.

In [1]:
import geopandas as gpd
import pandas as pd
import os
import glob

base_dir = "validation/2020_2021"

all_gdfs = []

# loop through each state folder
for state in os.listdir(base_dir):
    state_path = os.path.join(base_dir, state)
    if not os.path.isdir(state_path):
        continue

    # find all .shp files inside this state folder (could be nested too)
    shp_files = glob.glob(os.path.join(state_path, "**", "*.shp"), recursive=True)

    for shp_file in shp_files:
        try:
            gdf = gpd.read_file(shp_file)
            gdf["source_file"] = os.path.basename(shp_file)  # track origin for debugging
            gdf["source_state"] = state
            all_gdfs.append(gdf)
        except Exception as e:
            print(f"Failed to read {shp_file}: {e}")

print(f"Total shapefiles read: {len(all_gdfs)}")

Total shapefiles read: 284


In [2]:
# check all have the same CRS before combining — critical
crs_set = set(str(g.crs) for g in all_gdfs)
print(f"Unique CRS found: {crs_set}")

if len(crs_set) > 1:
    print("WARNING: multiple CRS found — need to reproject before combining")
else:
    drops_2021 = pd.concat(all_gdfs, ignore_index=True)
    drops_2021 = gpd.GeoDataFrame(drops_2021, geometry="geometry", crs=all_gdfs[0].crs)
    print(f"Combined total rows: {len(drops_2021)}")
    print(drops_2021.columns.tolist())

Unique CRS found: {'PROJCS["unknown",GEOGCS["unknown",DATUM["D_Unknown_based_on_GRS_1980_ellipsoid_using_towgs84_1_1_-1_0_0_0_0",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]'}
Combined total rows: 25499
['irwinid', 'IncNum1', 'Drp_ID1', 'new_pt', 'Vendor1', 'FalsPs1', 'SpltDr1', 'Latitd1', 'Longtd1', 'LoclDt1', 'LoclHr1', 'Knots1', 'HdngTr1', 'AltFet1', 'LoclTZ1', 'Gallns1', 'Tal_Nmb', 'UTCdtTm', 'lclMnth', 'loclDy1', 'loclYr1', 'LinLnM1', 'IncNam1', 'IADrop1', 'AC_Typ1', 'Fluid1', 'NONFIRE', 'Shp_Lng', 'DropMnt', 'DrpScnd', 'Shp_L_1', 'X', 'cntn_dt', 'Evnt_ID', 'buff_id', 'Shape_L

In [3]:
print(drops_2021["source_state"].value_counts())
print(drops_2021.columns.tolist())
print(drops_2021.head())

source_state
CA    10227
AZ     4108
MT     2217
CO     1886
WA     1884
OR     1810
NV     1158
ID      922
WY      406
UT      285
TX      232
MN      220
NM      100
NE       38
ND        6
Name: count, dtype: int64
['irwinid', 'IncNum1', 'Drp_ID1', 'new_pt', 'Vendor1', 'FalsPs1', 'SpltDr1', 'Latitd1', 'Longtd1', 'LoclDt1', 'LoclHr1', 'Knots1', 'HdngTr1', 'AltFet1', 'LoclTZ1', 'Gallns1', 'Tal_Nmb', 'UTCdtTm', 'lclMnth', 'loclDy1', 'loclYr1', 'LinLnM1', 'IncNam1', 'IADrop1', 'AC_Typ1', 'Fluid1', 'NONFIRE', 'Shp_Lng', 'DropMnt', 'DrpScnd', 'Shp_L_1', 'X', 'cntn_dt', 'Evnt_ID', 'buff_id', 'Shape_Leng', 'Shape_Area', 'geometry', 'source_file', 'source_state']
                                irwinid            IncNum1  \
0  7C5C61EA-CE99-4F35-A752-A5529C7ECB76  2020-UTSCS-200425   
1  7C5C61EA-CE99-4F35-A752-A5529C7ECB76  2020-UTSCS-200425   
2  7C5C61EA-CE99-4F35-A752-A5529C7ECB76  2020-UTSCS-200425   
3  7C5C61EA-CE99-4F35-A752-A5529C7ECB76  2020-UTSCS-200425   
4  7C5C61EA-CE99-4F35-A

In [4]:
drops_2021.to_file("validation/drops_2021_combined.geojson", driver="GeoJSON")
print(f"Saved {len(drops_2021)} rows to validation/drops_2021_combined.geojson")

Saved 25499 rows to validation/drops_2021_combined.geojson


In [5]:
drops_2021.to_file("validation/drops_2021_combined.shp")

/var/folders/gp/hgxpjqv54rg2r22rtswcl76r0000gn/T/ipykernel_8462/3499793787.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  drops_2021.to_file("validation/drops_2021_combined.shp")
/Users/sulavsapkota/miniforge3/envs/firenv/lib/python3.11/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'source_file' to 'source_fil'
  ogr_write(
/Users/sulavsapkota/miniforge3/envs/firenv/lib/python3.11/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'source_state' to 'source_sta'
  ogr_write(


---
## Part 2: Data Preparation

Same steps as Notebook 1 (CRS fix, date conversion, `drop_id` construction, DOB raster matching), applied to the combined 2020-2021 drops.

In [6]:
# ---- Imports ----
import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
import os
from rasterio.mask import mask
from shapely.ops import split as shapely_split, linemerge
from shapely.geometry import LineString, box

In [7]:
# ---- Load Combined 2020-2021 Drops ----
drops_2021 = gpd.read_file("validation/drops_2021_combined.shp")

print(f"Total rows: {len(drops_2021)}")
print(drops_2021.columns.tolist())
print(drops_2021.crs)

Total rows: 25499
['irwinid', 'IncNum1', 'Drp_ID1', 'new_pt', 'Vendor1', 'FalsPs1', 'SpltDr1', 'Latitd1', 'Longtd1', 'LoclDt1', 'LoclHr1', 'Knots1', 'HdngTr1', 'AltFet1', 'LoclTZ1', 'Gallns1', 'Tal_Nmb', 'UTCdtTm', 'lclMnth', 'loclDy1', 'loclYr1', 'LinLnM1', 'IncNam1', 'IADrop1', 'AC_Typ1', 'Fluid1', 'NONFIRE', 'Shp_Lng', 'DropMnt', 'DrpScnd', 'Shp_L_1', 'X', 'cntn_dt', 'Evnt_ID', 'buff_id', 'Shape_Leng', 'Shape_Area', 'source_fil', 'source_sta', 'geometry']
PROJCS["unknown",GEOGCS["unknown",DATUM["D_Unknown_based_on_GRS_1980_ellipsoid_using_towgs84_1_1_-1_0_0_0_0",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",E

In [8]:
drops_2021 = drops_2021.set_crs("EPSG:5070", allow_override=True)
print(drops_2021.crs)

EPSG:5070


In [9]:
drops_2021["local_date"] = pd.to_datetime(
    dict(
        year=drops_2021["loclYr1"].astype(int),
        month=drops_2021["lclMnth"].astype(int),
        day=drops_2021["loclDy1"].astype(int)
    ),
    errors="coerce"
)
drops_2021["drop_doy"] = drops_2021["local_date"].dt.dayofyear

print(drops_2021[["loclYr1", "lclMnth", "loclDy1", "drop_doy"]].head())
print(f"Missing drop_doy: {drops_2021['drop_doy'].isna().sum()}")

   loclYr1  lclMnth  loclDy1  drop_doy
0   2020.0      7.0     31.0       213
1   2020.0      7.0     31.0       213
2   2020.0      7.0     31.0       213
3   2020.0      7.0     31.0       213
4   2020.0      7.0     31.0       213
Missing drop_doy: 0


In [10]:
drops_2021["drop_id"] = (
    drops_2021["Evnt_ID"].astype(str) + "_" +
    drops_2021["Latitd1"].round(5).astype(str) + "_" +
    drops_2021["Longtd1"].round(5).astype(str)
)

print(drops_2021.groupby("drop_id").size().value_counts())

2    12694
4       24
1        9
6        1
Name: count, dtype: int64


In [11]:
import os
print(os.listdir("validation/"))

['drops_2021_combined.prj', '.DS_Store', 'drops_2021_combined.geojson', 'drops_2021_combined.cpg', 'drops_2021_combined.shp', 'drops_2021_combined.shx', 'DOB', '2020_2021', '2020_2021_backup', 'drops_2021_combined.dbf']


In [12]:
tif_dir = "validation/DOB"

# only grab actual .tif files, ignore .tif.aux.xml sidecar files
tif_files = {
    f.replace(".tif", "").replace("_1_dob", ""): f 
    for f in os.listdir(tif_dir) 
    if f.endswith(".tif") and not f.endswith(".tif.aux.xml")
}

print(f"Number of DOB rasters found: {len(tif_files)}")
print(list(tif_files.items())[:5])

Number of DOB rasters found: 358
[('CO4099510897320210620', 'CO4099510897320210620_1_dob.tif'), ('MT4569810891520200802', 'MT4569810891520200802_1_dob.tif'), ('MT4541710565520201006', 'MT4541710565520201006_1_dob.tif'), ('AZ3415011213920200531', 'AZ3415011213920200531_1_dob.tif'), ('MT4574510679520210719', 'MT4574510679520210719_1_dob.tif')]


In [13]:
drops_2021["Evnt_ID_str"] = drops_2021["Evnt_ID"].astype(str)
matched_mask = drops_2021["Evnt_ID_str"].isin(tif_files.keys())

print(f"Matched: {matched_mask.sum()} / {len(drops_2021)}")
print(f"Unmatched: {(~matched_mask).sum()}")

Matched: 25499 / 25499
Unmatched: 0


In [14]:
def get_tif_path(event_id):
    fname = tif_files.get(str(event_id))
    if fname is None:
        return None
    return os.path.join(tif_dir, fname)

drops_2021["tif_path"] = drops_2021["Evnt_ID_str"].apply(get_tif_path)

missing = drops_2021["tif_path"].isna().sum()
print(f"Dropping {missing} rows with no matching raster")
drops_2021 = drops_2021.dropna(subset=["tif_path"])
print(f"Remaining rows: {len(drops_2021)}")

Dropping 0 rows with no matching raster
Remaining rows: 25499


---
## Part 3: Feature Extraction -- DOB + Raster Split

Same extraction functions as Notebook 2, Parts 1 and 3, applied to the 2020-2021 drops.

In [15]:
# Open the tif file of the fire and extract pixels under the drop polygon. 
def extract_pixel_stats(geometry, tif_path, drop_doy, nodata_val=65535):
    try:
        with rasterio.open(tif_path) as src:
            #crop=True will prevent from loading the whole raster but only the polygon under the drop. 
            #filled=True will make sure that the pixels inside the bounding box but outside the drop polygon are set to nodata, so that only the pixels inside the polygon are counted as valid burn-day values. 
            # nodata_val=65535 - this variable represents the pixels which never burnt. 
            out_image, _ = mask(src, [geometry], crop=True, filled=True, nodata=nodata_val)

            #usually returns (band, cols, rows). since it is 2d, we will take [0]

            pixels = out_image[0]

    except Exception as e:
        return None
    


    # filter out pixels that burned. valid is a 1D array that has the doy of the burns. 
    valid = pixels[pixels != nodata_val]



    # This is for the edge cases where no pixels under the polygons burned. return NaN for DOB stats and 0 for all pixels counts. unburned_pixels and total_pixels are still meaningful. 
    if len(valid) == 0:
        return {
            "mean_dob" : np.nan, "median_dob":np.nan, 
            "early_pixels": 0, "late_pixels":0,
            "unburned_pixels": int(np.sum(pixels == nodata_val)), 
            "total_pixels" : int(pixels.size), "burned_pixels":0
        }
    

    # Normal case where at least some pixels under the drop burned. 
    return {
        # average and median burn day across all pixels. 
        "mean_dob": float(np.mean(valid)), 
        "median_dob": float(np.median(valid)),

        # pixels that burned before the drop date. Fire was already here (origin side)
        "early_pixels": int(np.sum(valid < drop_doy)),

        # pixels that burned after the drop date. Fire arrived here later. (protected side)
        "late_pixels": int(np.sum(valid > drop_doy)),
        
        # pixels that never burned at all
        "unburned_pixels" : int(np.sum(pixels == nodata_val)),

        #total pixels in the cropped bounding box (burned+unburned+boundary)
        "total_pixels": int(pixels.size),

        # total burned pixels 
        "burned_pixels": int(len(valid))
    }

In [16]:
# ---- Feature Extraction Loop ---- 
# Iterates over every drop-side row in the shapefile. 
# For each row, samples the DOB raster pixels under that polygon and computes burn-day statistics relative to the burn date. 
# Results are collected in a list of dicts, one per drop side, which is later converted into the training DataFrame. 

rows = [] # Stores one dict per successfully processed drop-side. 
skipped = 0 # tracks rows where extraction failed. 

for idx, drop in drops_2021.iterrows():

    # Extract pixel statistics from the DOB raster under this drop-side polygon. Returns None if any error occurs. 
    stats = extract_pixel_stats(
        geometry=drop.geometry, # polygon shape of the drop side.
        tif_path = drop["tif_path"], # path to this fire's DOB raster
        drop_doy = drop["drop_doy"], # drop date as day-of-year for early/late pixel comparison. 

    )


    # skip this row if extraction failed and count it. 
    if stats is None:
        skipped += 1
        continue


    # Attach identifying and operational metadata from the shapefile row to the raster-derived stats, so each row in the final DataFrame contains both feature values and bookkeeping information. 

    stats.update({
        "event_id": drop["Evnt_ID"], # fire incident ID. Later used for GroupKFold splitting. 
        "buff_id": drop["buff_id"], # which side of the drop (1 or 2)
        "altitude": drop.get("altitude", np.nan), # aircraft altitude at the time of drop
        "knots": drop.get("Knots1", np.nan), # aircraft speed at the time of drop
        "gallons": drop.get("Gallns1", np.nan), # volume of retardants dropped
        "drop_seconds": drop.get("DrpScnd", np.nan), # duration of drop in seconds
        "pt": drop["new_pt"], # target variable : 1 = protected side, 0 = origin side. 
        "drop_id" : drop["drop_id"] # unique drop identifier (event_id + lat + long + utcdate) - used to pair two sides of the same drop. 
    })

    rows.append(stats)


# BUILDING THE DATAFRAME

print(f"Processed: {len(rows)}, Skipped: {skipped}")

# Convert list of dicts to DataFrame - one row per drop-side
df_dob = pd.DataFrame(rows)

# Saving the file to CSV so features don't need to be extracted every time. 
df_dob.to_csv("training_features_dob_2021.csv", index=False)
print(df_dob.head())

Processed: 24585, Skipped: 914
   mean_dob  median_dob  early_pixels  late_pixels  unburned_pixels  \
0     213.0       213.0             0            0               31   
1     213.0       213.0             0            0               66   
2       NaN         NaN             0            0               54   
3     213.0       213.0             0            0               51   
4     213.0       213.0             0            0              145   

   total_pixels  burned_pixels               event_id  buff_id  altitude  \
0            34              3  UT3979911153620200731      1.0       NaN   
1           180            114  UT3979911153620200731      2.0       NaN   
2            54              0  UT3979911153620200731      1.0       NaN   
3            88             37  UT3979911153620200731      2.0       NaN   
4           196             51  UT3979911153620200731      1.0       NaN   

   knots  gallons drop_seconds  pt                                    drop_id  
0  15

In [17]:
# Running some sanity check to see if our training data shows real signal and the extracted features actually differ between the two classes (protected vs origin)

# If features carry a real signal, pt=1 (protected) should show: 
# - more unburned pixels (protected side less likely to have burned)
# - fewer early pixels (less fire activity before the drop on the protected side)
# - fewer burned pixels (less total fire)

# If the means look nearly identical between pt=0 and pt=1, the features have no discriminative power and model won't learn anything useful. 

df_dob.groupby("pt")[["unburned_pixels", "early_pixels", "late_pixels", "burned_pixels"]].mean()

,unburned_pixels,early_pixels,late_pixels,burned_pixels
pt,,,,
0,196.626272,16.275715,30.084289,98.285564
1,226.116495,4.548942,29.035170,59.546483
5,146.000000,9.000000,0.000000,10.000000


In [18]:
# ---- Extend Drop Line Function ----
from shapely.ops import split as shapely_split, linemerge
from shapely.geometry import LineString, box

def extend_line(line, distance=500000):
    """
    Extend a LineString in both directions far beyond raster bounds
    (distance=500km ensures the line cuts through any CONUS fire raster).
    Handles MultiLineString by merging into a single line first.
    """
    # handle MultiLineString — merge into single LineString
    if line.geom_type == "MultiLineString":
        line = linemerge(line)
        # if merge still gives MultiLineString, take the longest segment
        if line.geom_type == "MultiLineString":
            line = max(line.geoms, key=lambda g: g.length)

    coords = list(line.coords)
    start = np.array(coords[0])
    end = np.array(coords[-1])

    # unit direction vector of the line
    direction = end - start
    direction = direction / np.linalg.norm(direction)

    # extend both endpoints far beyond raster bounds
    new_start = start - direction * distance
    new_end = end + direction * distance

    return LineString([new_start, new_end])

In [19]:
# ---- Split Extraction Function ----

def extract_split_features(pair, tif_path, nodata_val=65535):
    """
    Split the DOB raster by the drop line and extract pixel stats
    for each half, matched to the correct buff_id.

    pair: GeoDataFrame with exactly 2 rows (both sides of one drop)
    tif_path: path to the fire's DOB raster

    Returns (stats_side1, stats_side2) or None if extraction fails.
    """
    row_1 = pair[pair["buff_id"] == 1].iloc[0]
    row_2 = pair[pair["buff_id"] == 2].iloc[0]
    geom_1 = row_1.geometry
    geom_2 = row_2.geometry

    # get shared boundary between the two polygons = the drop line
    drop_line = geom_1.intersection(geom_2)

    # guard: skip if no shared boundary
    if drop_line.is_empty:
        return None

    # guard: handle GeometryCollection by extracting line components
    if drop_line.geom_type == "GeometryCollection":
        lines = [g for g in drop_line.geoms if "Line" in g.geom_type]
        if not lines:
            return None
        drop_line = linemerge(lines)

    # guard: must be a line geometry at this point
    if drop_line.geom_type not in ["LineString", "MultiLineString"]:
        return None

    # extend the line far beyond the raster bounds
    extended = extend_line(drop_line)

    with rasterio.open(tif_path) as src:
        bounds = src.bounds
        raster_bbox = box(bounds.left, bounds.bottom, bounds.right, bounds.top)

        # split the raster's bounding box into two halves using the extended line
        try:
            result = shapely_split(raster_bbox, extended)
            if len(result.geoms) != 2:
                return None
            half_A, half_B = result.geoms
        except Exception:
            return None

        # match halves to buff_ids using polygon centroids
        # (the split doesn't know which half is 'side 1' vs 'side 2')
        centroid_1 = geom_1.centroid
        if half_A.contains(centroid_1):
            side1_geom, side2_geom = half_A, half_B
        else:
            side1_geom, side2_geom = half_B, half_A

        # sample pixels for each side
        try:
            img_1, _ = mask(src, [side1_geom], crop=True, filled=True, nodata=nodata_val)
            img_2, _ = mask(src, [side2_geom], crop=True, filled=True, nodata=nodata_val)
        except Exception:
            return None

    pixels_1 = img_1[0]
    pixels_2 = img_2[0]

    def compute_stats(pixels, drop_doy, nodata_val):
        valid = pixels[pixels != nodata_val]
        if len(valid) == 0:
            return {
                "split_mean_dob": np.nan,
                "split_early_pixels": 0,
                "split_late_pixels": 0,
                "split_unburned_pixels": int(np.sum(pixels == nodata_val)),
                "split_burned_pixels": 0,
                "split_total_pixels": int(pixels.size)
            }
        return {
            "split_mean_dob":        float(np.mean(valid)),
            "split_early_pixels":    int(np.sum(valid < drop_doy)),
            "split_late_pixels":     int(np.sum(valid > drop_doy)),
            "split_unburned_pixels": int(np.sum(pixels == nodata_val)),
            "split_burned_pixels":   int(len(valid)),
            "split_total_pixels":    int(pixels.size)
        }

    stats_1 = compute_stats(pixels_1, row_1["drop_doy"], nodata_val)
    stats_2 = compute_stats(pixels_2, row_2["drop_doy"], nodata_val)

    stats_1.update({"event_id": row_1["Evnt_ID"], "buff_id": 1, "drop_id": row_1["drop_id"]})
    stats_2.update({"event_id": row_2["Evnt_ID"], "buff_id": 2, "drop_id": row_2["drop_id"]})

    return stats_1, stats_2

In [20]:
# ---- Split Extraction Loop ----
# Process one drop pair at a time (need both sides to construct the drop line)

split_rows = []
skipped_split = 0

drop_pairs = drops_2021.groupby("drop_id")

for drop_id, pair in drop_pairs:
    # only process clean pairs (exactly 2 sides, both buff_ids present)
    if len(pair) != 2:
        skipped_split += 1
        continue
    if set(pair["buff_id"].astype(int)) != {1, 2}:
        skipped_split += 1
        continue

    tif_path = pair.iloc[0]["tif_path"]
    result = extract_split_features(pair, tif_path)

    if result is None:
        skipped_split += 1
        continue

    stats_1, stats_2 = result
    split_rows.append(stats_1)
    split_rows.append(stats_2)

print(f"Processed: {len(split_rows)} rows ({len(split_rows)//2} drop pairs)")
print(f"Skipped: {skipped_split}")

df_split_2021 = pd.DataFrame(split_rows)
df_split_2021.to_csv("training_features_split_2021.csv", index=False)
print(df_split_2021.head())

Processed: 25152 rows (12576 drop pairs)
Skipped: 152
   split_mean_dob  split_early_pixels  split_late_pixels  \
0      161.240702                   0               1029   
1      161.000000                   0                  0   
2      161.217179                   0                579   
3      161.203989                   0                450   
4      161.254766                   0               1029   

   split_unburned_pixels  split_burned_pixels  split_total_pixels  \
0                   3723                 4275                7998   
1                   1983                  597                2580   
2                   4386                 2666                7052   
3                   4416                 2206                6622   
4                   3959                 4039                7998   

                event_id  buff_id                                    drop_id  
0  AZ3140011075320210610        1  AZ3140011075320210610_31.39804_-110.74332  
1  AZ3140011

In [21]:
import numpy as np
from scipy.ndimage import distance_transform_edt
from shapely.geometry import Point
import rasterio

def compute_dist_to_unburned(tif_path, centroid_1, centroid_2, nodata_val=65535):
    """
    For each polygon centroid, compute distance to nearest unburned pixel.
    Protected side should be CLOSER to unburned area.
    Origin side should be FURTHER from unburned area (surrounded by fire).
    """
    try:
        with rasterio.open(tif_path) as src:
            arr = src.read(1)
            pixel_size = src.res[0]  # meters per pixel

            # binary mask: True = burned, False = unburned
            burned_mask = (arr != nodata_val).astype(np.uint8)

            # distance transform on burned mask
            # gives each burned pixel its distance to nearest unburned pixel
            dist_to_unburned = distance_transform_edt(burned_mask) * pixel_size

            # sample distance at each polygon centroid
            row_1, col_1 = src.index(centroid_1.x, centroid_1.y)
            row_2, col_2 = src.index(centroid_2.x, centroid_2.y)

            # clamp to raster bounds just in case
            row_1 = np.clip(row_1, 0, arr.shape[0]-1)
            col_1 = np.clip(col_1, 0, arr.shape[1]-1)
            row_2 = np.clip(row_2, 0, arr.shape[0]-1)
            col_2 = np.clip(col_2, 0, arr.shape[1]-1)

            dist_1 = float(dist_to_unburned[row_1, col_1])
            dist_2 = float(dist_to_unburned[row_2, col_2])

            return dist_1, dist_2

    except Exception as e:
        return None, None

In [22]:
from tqdm import tqdm

dist_rows = []
skipped = 0

drop_pairs = drops_2021.groupby("drop_id")
drop_pairs_list = list(drops_2021.groupby("drop_id"))


for drop_id, pair in tqdm(drop_pairs_list, desc="Distance extraction"):
    if len(pair) != 2:
        skipped += 1
        continue
    if set(pair["buff_id"].astype(int)) != {1, 2}:
        skipped += 1
        continue

    tif_path = pair.iloc[0]["tif_path"]
    
    row_1 = pair[pair["buff_id"] == 1].iloc[0]
    row_2 = pair[pair["buff_id"] == 2].iloc[0]
    
    centroid_1 = row_1.geometry.centroid
    centroid_2 = row_2.geometry.centroid

    dist_1, dist_2 = compute_dist_to_unburned(
        tif_path, centroid_1, centroid_2
    )

    if dist_1 is None:
        skipped += 1
        continue

    dist_rows.append({
        "event_id": row_1["Evnt_ID"],
        "buff_id": 1,
        "drop_id": drop_id,
        "dist_to_unburned": dist_1
    })
    dist_rows.append({
        "event_id": row_2["Evnt_ID"],
        "buff_id": 2,
        "drop_id": drop_id,
        "dist_to_unburned": dist_2
    })

print(f"Processed: {len(dist_rows)} rows, Skipped: {skipped}")
df_dist_2021 = pd.DataFrame(dist_rows)
df_dist_2021.to_csv("training_features_dist_2021.csv", index=False)

Distance extraction: 100%|██████████| 12728/12728 [12:37<00:00, 16.80it/s] 


Processed: 25388 rows, Skipped: 34


In [23]:
def extract_expanded_polygon_features(pair, tif_path, 
                                       radius_fraction=0.25, 
                                       nodata_val=65535):
    """
    Instead of splitting the entire raster, expand each drop polygon
    to cover ~25% of the raster diagonal on its respective side.
    This avoids capturing irrelevant fire arms from complex fire shapes.
    """
    row_1 = pair[pair["buff_id"] == 1].iloc[0]
    row_2 = pair[pair["buff_id"] == 2].iloc[0]
    geom_1 = row_1.geometry
    geom_2 = row_2.geometry
    drop_doy = row_1["drop_doy"]

    drop_line = geom_1.intersection(geom_2)
    if drop_line.is_empty:
        return None

    if drop_line.geom_type == "GeometryCollection":
        from shapely.ops import linemerge
        lines = [g for g in drop_line.geoms if "Line" in g.geom_type]
        if not lines:
            return None
        drop_line = linemerge(lines)

    if drop_line.geom_type not in ["LineString", "MultiLineString"]:
        return None

    extended = extend_line(drop_line)

    with rasterio.open(tif_path) as src:
        bounds = src.bounds

        # compute raster diagonal → determines expansion radius
        raster_diagonal = ((bounds.right - bounds.left)**2 + 
                           (bounds.top - bounds.bottom)**2) ** 0.5
        radius = raster_diagonal * radius_fraction

        # split raster bounding box
        raster_bbox = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
        try:
            result = shapely_split(raster_bbox, extended)
            if len(result.geoms) != 2:
                return None
            half_A, half_B = result.geoms
        except Exception:
            return None

        # match halves to buff_ids
        centroid_1 = geom_1.centroid
        if half_A.contains(centroid_1):
            side1_geom, side2_geom = half_A, half_B
        else:
            side1_geom, side2_geom = half_B, half_A

        # expand: intersect each half with a circle around drop center
        drop_center = drop_line.centroid
        local_buffer = drop_center.buffer(radius)

        side1_local = side1_geom.intersection(local_buffer)
        side2_local = side2_geom.intersection(local_buffer)

        if side1_local.is_empty or side2_local.is_empty:
            return None

        # sample pixels in expanded local areas
        try:
            img_1, _ = mask(src, [side1_local], crop=True, 
                           filled=True, nodata=nodata_val)
            img_2, _ = mask(src, [side2_local], crop=True, 
                           filled=True, nodata=nodata_val)
        except Exception:
            return None

    pixels_1 = img_1[0]
    pixels_2 = img_2[0]

    def compute_stats(pixels, drop_doy, nodata_val):
        valid = pixels[pixels != nodata_val]
        if len(valid) == 0:
            return {
                "exp_mean_dob": np.nan,
                "exp_burned_pixels": 0,
                "exp_unburned_pixels": int(np.sum(pixels == nodata_val)),
                "exp_early_pixels": 0,
                "exp_late_pixels": 0,
                "exp_total_pixels": int(pixels.size)
            }
        return {
            "exp_mean_dob":       float(np.mean(valid)),
            "exp_burned_pixels":  int(len(valid)),
            "exp_unburned_pixels": int(np.sum(pixels == nodata_val)),
            "exp_early_pixels":   int(np.sum(valid < drop_doy)),
            "exp_late_pixels":    int(np.sum(valid > drop_doy)),
            "exp_total_pixels":   int(pixels.size)
        }

    stats_1 = compute_stats(pixels_1, drop_doy, nodata_val)
    stats_2 = compute_stats(pixels_2, drop_doy, nodata_val)

    stats_1.update({"event_id": row_1["Evnt_ID"], 
                    "buff_id": 1, 
                    "drop_id": row_1["drop_id"]})
    stats_2.update({"event_id": row_2["Evnt_ID"], 
                    "buff_id": 2, 
                    "drop_id": row_2["drop_id"]})

    return stats_1, stats_2

In [24]:
from tqdm import tqdm

exp_rows = []
skipped = 0

drop_pairs_list = list(drops_2021.groupby("drop_id"))

for drop_id, pair in tqdm(drop_pairs_list, desc="Expanded polygon extraction"):
    if len(pair) != 2:
        skipped += 1
        continue
    if set(pair["buff_id"].astype(int)) != {1, 2}:
        skipped += 1
        continue

    tif_path = pair.iloc[0]["tif_path"]
    result = extract_expanded_polygon_features(pair, tif_path)

    if result is None:
        skipped += 1
        continue

    stats_1, stats_2 = result
    exp_rows.append(stats_1)
    exp_rows.append(stats_2)

print(f"Processed: {len(exp_rows)} rows, Skipped: {skipped}")
df_exp_2021 = pd.DataFrame(exp_rows)
df_exp_2021.to_csv("training_features_expanded_2021.csv", index=False)

Expanded polygon extraction: 100%|██████████| 12728/12728 [04:14<00:00, 49.92it/s] 


Processed: 25108 rows, Skipped: 174


---
## Part 4: Merge, Clean, and Predict

In [25]:
df_2021 = df_dob.merge(df_split_2021, on=["event_id","buff_id","drop_id"], how="left")
df_2021 = df_2021.merge(df_dist_2021, on=["event_id","buff_id","drop_id"], how="left")
df_2021 = df_2021.merge(df_exp_2021,  on=["event_id","buff_id","drop_id"], how="left")


In [26]:
df_2021 = df_2021[df_2021["pt"].isin([0, 1])]
print(f"Rows after removing invalid pt: {len(df_2021)}")

Rows after removing invalid pt: 24584


In [27]:
# ---- Prepare Base DataFrame ----
group_sizes = df_2021.groupby("drop_id").size()
clean_drop_ids = group_sizes[group_sizes == 2].index
df_model = df_2021[df_2021["drop_id"].isin(clean_drop_ids)].copy()
print(f"Clean pairs: {len(df_model)} rows ({df_model['drop_id'].nunique()} drops)")

# Fix NaN mean_dob/median_dob/split_mean_dob
df_model["mean_dob"]       = df_model["mean_dob"].fillna(0)
df_model["median_dob"]     = df_model["median_dob"].fillna(0)
df_model["split_mean_dob"] = df_model["split_mean_dob"].fillna(0)

# ---- Build Paired-Difference Features (all on df_model, once each) ----

# DOB
for col in ["unburned_pixels", "early_pixels", "burned_pixels"]:
    df_model[f"{col.replace('_pixels', '')}_pair_diff"] = \
        df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())

# Split
for col in ["split_unburned_pixels", "split_early_pixels", "split_burned_pixels"]:
    new_col = col.replace("_pixels", "") + "_pair_diff"
    df_model[new_col] = df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())

# Distance
df_model["dist_pair_diff"] = df_model.groupby("drop_id")["dist_to_unburned"].transform(
    lambda x: x - x.mean()
)

# Expanded polygon
for col in ["exp_burned_pixels", "exp_unburned_pixels", "exp_early_pixels"]:
    new_col = col.replace("_pixels", "") + "_pair_diff"
    df_model[new_col] = df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())

print(f"DataFrame shape after feature engineering: {df_model.shape}")

Clean pairs: 24296 rows (12148 drops)
DataFrame shape after feature engineering: (24296, 38)


In [35]:
import joblib
final_features_v2 = joblib.load("./refined/model_features_v2.pkl")
missing = [f for f in final_features_v2 if f not in df_model.columns]
print(f"Missing columns: {missing}")

Missing columns: []


In [36]:
import joblib
model_final = joblib.load("refined/model_final_v2.pkl")
final_features = joblib.load("refined/model_features_v2.pkl")

In [37]:
from sklearn.metrics import accuracy_score

df_2021_clean = df_model.dropna(subset=final_features + ["pt"])
preds = model_final.predict(df_2021_clean[final_features])
acc = accuracy_score(df_2021_clean["pt"], preds)
print(acc)

0.8959084430284229


In [36]:
# Defining Feature Sets

raw_features = [
    "mean_dob", "median_dob", "early_pixels", "late_pixels",
    "unburned_pixels", "burned_pixels", "total_pixels"
]

diff_features = [
    "unburned_pair_diff", "early_pair_diff", "burned_pair_diff"
]

dob_features = raw_features + diff_features

fuel_features = [
    "dominant_fuel", "fuel_diversity",
    "grass_pixels", "shrub_pixels", "timber_pixels",
    "slash_pixels", "nonburnable_pixels",
    "grass_pair_diff", "shrub_pair_diff",
    "timber_pair_diff", "nonburnable_pair_diff"
]

# Whole-raster split features — the strongest predictor discovered.
# Same statistics as DOB features, but computed across the ENTIRE fire raster
# split by the extended drop line, rather than just the small buffer polygon.
split_features = [
    "split_mean_dob", "split_early_pixels", "split_late_pixels",
    "split_unburned_pixels", "split_burned_pixels", "split_total_pixels",
    "split_burned_pair_diff", "split_unburned_pair_diff", "split_early_pair_diff"
]


dist_features = [
    "dist_to_unburned", "dist_pair_diff"
]


all_features = dob_features + fuel_features + split_features


# New fix after error analysis

agreement_features = [
    "signal_agreement",
    "local_signal_strength",
    "split_signal_strength",
    "split_dominance"
]

new_features = dob_features + split_features + [
    "dist_to_unburned", 
    "dist_pair_diff"
]


In [ ]:
partial_features = dob_features + split_features + dist_features
df_model_clean2 = df_model.dropna(subset=partial_features + ["pt"])
